# 02 - Weather Integration & Feature Engineering
**Person A: Data, Weather, and Features**

### Objectives:
1. Merge historical weather data (wind speed, solar irradiation, temperature, cloud cover) from NASA POWER.
2. Build time-series lag features (1h, 24h, 168h).
3. Compute rolling window statistics (24h and 168h mean and std).
4. Extract calendar/cyclical features (hour, day of week, month, is_weekend).
5. Perform chronological train/test split and export processed CSVs.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loading import load_raw, clean, merge_weather, chronological_split, save_processed
from features import build_features
from plotting import apply_style, savefig, COLORS

apply_style()

## 1. Load Data and Merge Historical Weather

In [ ]:
df_raw = load_raw()
df_clean = clean(df_raw)
df_merged = merge_weather(df_clean)

print(f"Cleaned shape: {df_clean.shape}")
print(f"Merged shape with weather: {df_merged.shape}")
df_merged.head()

## 2. Feature Engineering (Lags, Rolling Stats & Calendar)

In [ ]:
df_features = build_features(df_merged)
print(f"Feature matrix shape: {df_features.shape}")
print(f"Columns ({len(df_features.columns)}):\n{df_features.columns.tolist()}")
df_features.head()

## 3. Feature Correlation Matrix

In [ ]:
numeric_cols = df_features.select_dtypes(include='number').columns
corr = df_features[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
savefig('../results/figures/02_correlation.png')
plt.show()

## 4. Lag Features vs Production Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
lags = [('lag_1h', 'Lag 1 Hour (Autocorrelation)'), 
        ('lag_24h', 'Lag 24 Hours (Daily Seasonality)'), 
        ('lag_168h', 'Lag 168 Hours (Weekly Seasonality)')]

sample = df_features.sample(min(5000, len(df_features)), random_state=42)
for ax, (col, title) in zip(axes, lags):
    ax.scatter(sample[col], sample['Production'], alpha=0.25, s=8, color='#4A90E2')
    ax.set_title(title)
    ax.set_xlabel(f'{col} (MWh)')
    ax.set_ylabel('Production (MWh)')
plt.tight_layout()
savefig('../results/figures/02_lag_scatter.png')
plt.show()

## 5. Weather Features vs Production

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
weather_vars = [
    ('wind_speed_10m', 'Wind Speed at 10m (m/s)', 'Wind Speed vs Production'),
    ('irradiation', 'Global Solar Irradiation (kW-hr/m2/day)', 'Solar Irradiation vs Production'),
    ('temp_2m', 'Temperature at 2m (C)', 'Temperature vs Production'),
    ('cloud_cover', 'Cloud Cover (%)', 'Cloud Cover vs Production')
]

for ax, (col, xlab, title) in zip(axes.flatten(), weather_vars):
    ax.scatter(sample[col], sample['Production'], alpha=0.25, s=8, color='#F5A623' if 'irrad' in col else '#50E3C2')
    ax.set_title(title)
    ax.set_xlabel(xlab)
    ax.set_ylabel('Production (MWh)')
plt.tight_layout()
savefig('../results/figures/02_weather_scatter.png')
plt.show()

## 6. Chronological Train / Test Split and Export

In [ ]:
train, test = chronological_split(df_features, test_ratio=0.2)
print(f"Total rows: {len(df_features)}")
print(f"Train split: {len(train)} rows ({train.index.min()} to {train.index.max()})")
print(f"Test split:  {len(test)} rows ({test.index.min()} to {test.index.max()})")

save_processed(df_features, 'features_all.csv')
save_processed(train,       'train_features.csv')
save_processed(test,        'test_features.csv')
print("Successfully generated all processed dataset files for Person B and Person C!")